# How to Improve CNN Accuracy in PyTorch
### Learn with Dr. Jyoti | Hands-on Google Colab tutorial

**Goal:** Compare a baseline CNN with data augmentation and dropout on Fashion-MNIST.

We will:
1. Create a training/validation split and keep the test set separate.
2. Train a baseline CNN.
3. Try data augmentation.
4. Try dropout.
5. Compare validation results, evaluate the selected model on the test set, and inspect mistakes.

**Teaching note:** Accuracy improvements are not guaranteed. The goal is to learn how to run fair experiments and interpret results.

## 1. Imports and reproducibility
Run this cell first. The notebook uses PyTorch and torchvision, which are available in Google Colab.

In [ ]:
import copy
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## 2. Load Fashion-MNIST
Fashion-MNIST contains 28 × 28 grayscale clothing images in 10 categories. We normalize pixel values for training. We will use the official training split for training/validation and reserve the official test split for final evaluation.

In [ ]:
class_names = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

basic_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.FashionMNIST(
    root="./data", train=True, download=True, transform=basic_transform
)
test_dataset = datasets.FashionMNIST(
    root="./data", train=False, download=True, transform=basic_transform
)

print("Original training images:", len(train_dataset))
print("Test images:", len(test_dataset))

In [ ]:
# Display a few sample images.
fig, axes = plt.subplots(1, 5, figsize=(12, 3))
for i, ax in enumerate(axes):
    image, label = train_dataset[i]
    ax.imshow(image.squeeze(0) * 0.5 + 0.5, cmap="gray")
    ax.set_title(class_names[label])
    ax.axis("off")
plt.tight_layout()
plt.show()

## 3. Make a training/validation split
We use 55,000 images for training and 5,000 for validation. The test set stays untouched until the final evaluation.

In [ ]:
from torch.utils.data import random_split

generator = torch.Generator().manual_seed(SEED)
train_subset, val_subset = random_split(
    train_dataset, [55000, 5000], generator=generator
)

train_loader = DataLoader(train_subset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print("Training images:", len(train_subset))
print("Validation images:", len(val_subset))
print("Test images:", len(test_dataset))

## 4. Define the baseline CNN
The CNN uses two convolution layers to learn image features. Max pooling reduces the spatial size, and the final linear layer produces scores for 10 clothing categories.

In [ ]:
class FashionCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 10)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

## 5. Create reusable training and evaluation functions
This function records training and validation loss and accuracy for every epoch. It saves the model state with the best validation accuracy. This avoids automatically choosing the last epoch.

In [ ]:
def train_model(model, train_loader, val_loader, epochs=5, learning_rate=0.001):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    history = {
        "train_loss": [], "train_accuracy": [],
        "val_loss": [], "val_accuracy": []
    }
    best_val_accuracy = -1.0
    best_state = copy.deepcopy(model.state_dict())

    for epoch in range(epochs):
        model.train()
        running_loss, train_correct, train_total = 0.0, 0, 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            train_correct += (outputs.argmax(dim=1) == labels).sum().item()
            train_total += labels.size(0)

        train_loss = running_loss / train_total
        train_acc = 100 * train_correct / train_total

        model.eval()
        val_loss_total, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss_total += loss.item() * images.size(0)
                val_correct += (outputs.argmax(dim=1) == labels).sum().item()
                val_total += labels.size(0)

        val_loss = val_loss_total / val_total
        val_acc = 100 * val_correct / val_total

        history["train_loss"].append(train_loss)
        history["train_accuracy"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_accuracy"].append(val_acc)

        if val_acc > best_val_accuracy:
            best_val_accuracy = val_acc
            best_state = copy.deepcopy(model.state_dict())

        print(
            f"Epoch {epoch + 1}/{epochs} | "
            f"Train loss: {train_loss:.4f} | Train accuracy: {train_acc:.2f}% | "
            f"Validation loss: {val_loss:.4f} | Validation accuracy: {val_acc:.2f}%"
        )

    model.load_state_dict(best_state)
    return history


def evaluate_accuracy(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            predictions = model(images).argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)
    return 100 * correct / total


def plot_history(history, title):
    epochs = range(1, len(history["train_loss"]) + 1)
    plt.figure(figsize=(7, 4))
    plt.plot(epochs, history["train_loss"], marker="o", label="Training loss")
    plt.plot(epochs, history["val_loss"], marker="o", label="Validation loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(title + " — Loss")
    plt.legend()
    plt.grid(True)
    plt.show()

    plt.figure(figsize=(7, 4))
    plt.plot(epochs, history["train_accuracy"], marker="o", label="Training accuracy")
    plt.plot(epochs, history["val_accuracy"], marker="o", label="Validation accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy (%)")
    plt.title(title + " — Accuracy")
    plt.legend()
    plt.grid(True)
    plt.show()

## 6. Experiment A — Baseline CNN
This is our reference model: no data augmentation and no dropout. The later experiments will use the same training/validation split and training settings.

In [ ]:
baseline_model = FashionCNN().to(device)
baseline_history = train_model(baseline_model, train_loader, val_loader, epochs=5)
plot_history(baseline_history, "Baseline CNN")

## 7. Experiment B — Data augmentation
Data augmentation creates slightly altered training images, such as images shifted or rotated a little. It can help the model handle reasonable visual variation. We apply random transformations only to training examples—not validation or test examples.

In [ ]:
augmented_transform = transforms.Compose([
    transforms.RandomAffine(degrees=10, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])
evaluation_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

augmented_full_train = datasets.FashionMNIST(
    root="./data", train=True, download=True, transform=augmented_transform
)
evaluation_full_train = datasets.FashionMNIST(
    root="./data", train=True, download=True, transform=evaluation_transform
)

# Reuse exactly the same indices for the split, but apply augmentation to training only.
augmented_train_subset = Subset(augmented_full_train, train_subset.indices)
evaluation_val_subset = Subset(evaluation_full_train, val_subset.indices)

augmented_train_loader = DataLoader(
    augmented_train_subset, batch_size=64, shuffle=True
)
evaluation_val_loader = DataLoader(
    evaluation_val_subset, batch_size=64, shuffle=False
)

print("Augmented training images:", len(augmented_train_subset))
print("Validation images:", len(evaluation_val_subset))

In [ ]:
# Visualize a few examples as they appear after random augmentation.
images, labels = next(iter(augmented_train_loader))
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    image = images[i].squeeze(0) * 0.5 + 0.5
    ax.imshow(image, cmap="gray")
    ax.set_title(class_names[labels[i].item()])
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
augmentation_model = FashionCNN().to(device)
augmentation_history = train_model(
    augmentation_model, augmented_train_loader, evaluation_val_loader, epochs=5
)
plot_history(augmentation_history, "CNN with Data Augmentation")

## 8. Experiment C — Dropout
Dropout randomly sets some activations to zero during training. Here, a dropout rate of 0.3 means each activation entering the dropout layer has a 30% chance of being zeroed during training. Dropout is disabled automatically when the model is in evaluation mode.

In [ ]:
class FashionCNNWithDropout(nn.Module):
    def __init__(self, dropout_rate=0.3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(32 * 7 * 7, 10)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

In [ ]:
dropout_model = FashionCNNWithDropout(dropout_rate=0.3).to(device)
dropout_history = train_model(dropout_model, train_loader, val_loader, epochs=5)
plot_history(dropout_history, "CNN with Dropout")

## 9. Compare the experiments
Compare the best validation accuracy for each model. Also inspect the learning curves: a single number cannot tell the whole story. Results can vary slightly across runtime and hardware.

In [ ]:
experiments = {
    "Baseline CNN": baseline_history,
    "Data augmentation": augmentation_history,
    "Dropout": dropout_history
}
models = {
    "Baseline CNN": baseline_model,
    "Data augmentation": augmentation_model,
    "Dropout": dropout_model
}

for name, history in experiments.items():
    print(f"{name}: best validation accuracy = {max(history['val_accuracy']):.2f}%")

## 10. Final test evaluation
Choose the model using validation accuracy, then evaluate it once on the held-out test set. Avoid repeatedly choosing models based on test accuracy.

In [ ]:
best_name = max(
    experiments,
    key=lambda name: max(experiments[name]["val_accuracy"])
)
best_model = models[best_name]
print("Selected model:", best_name)
print(f"Final test accuracy: {evaluate_accuracy(best_model, test_loader):.2f}%")

## 11. Inspect incorrect predictions
Looking at mistakes can reveal categories the model confuses. We inspect examples from the held-out test set only after selecting the model.

In [ ]:
def show_mistakes(model, dataset, count=10):
    model.eval()
    mistakes = []
    for index in range(len(dataset)):
        image, label = dataset[index]
        with torch.no_grad():
            output = model(image.unsqueeze(0).to(device))
            predicted = output.argmax(dim=1).item()
        if predicted != label:
            mistakes.append((image, label, predicted))
        if len(mistakes) >= count:
            break

    if not mistakes:
        print("No mistakes found among the inspected examples.")
        return

    fig, axes = plt.subplots(2, 5, figsize=(13, 6))
    for ax, (image, actual, predicted) in zip(axes.flat, mistakes):
        ax.imshow(image.squeeze(0) * 0.5 + 0.5, cmap="gray")
        ax.set_title(f"Predicted: {class_names[predicted]}\nActual: {class_names[actual]}")
        ax.axis("off")
    for ax in axes.flat[len(mistakes):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

show_mistakes(best_model, test_dataset, count=10)

## 12. Recap and discussion

- **Measure fairly:** use training and validation results to understand generalization.
- **Data augmentation:** adds realistic variation to training examples.
- **Dropout:** regularizes the model by randomly disabling activations during training.
- **Compare experiments:** no technique is guaranteed to improve accuracy.
- **Keep the test set separate:** use it for final evaluation after choosing a model.

**Try it yourself:** Change the augmentation strength or dropout rate, run the experiment again, and compare validation curves. Change one thing at a time.